# 🃏 The Mind — LLMs + Aprendizaje por Refuerzo

Simulamos el juego cooperativo **The Mind** con 4 agentes LLM que:
- Se comunican libremente (sin revelar sus cartas)
- Aprenden a coordinarse mediante RL (GRPO)
- Desarrollan lenguaje emergente propio

## Hardware recomendado
| Config | Modelo | Dispositivo |
|--------|--------|-------------|
| GPU 4GB | Qwen2.5-1.5B-Instruct + 4bit | cuda |
| CPU 12GB | Qwen2.5-1.5B-Instruct | cpu |
| GPU 4GB (ligero) | Qwen2.5-0.5B-Instruct | cuda |

In [1]:
# ── Instalación de dependencias ──────────────────────────────────────────────
# Ejecuta esta celda solo la primera vez

# !pip install torch transformers peft accelerate bitsandbytes matplotlib tqdm

# Opcional: Unsloth para entrenar más rápido (GPU con CUDA)
# !pip install "unsloth[cu121-ampere-torch240] @ git+https://github.com/unslothai/unsloth.git"

# trl para utilidades adicionales de RL (opcional)
# !pip install trl

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import os
import torch
import warnings

warnings.filterwarnings('ignore')  # para evitar mensajes de advertencia molestos

# Añade el directorio del proyecto al path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from environment import TheMindEnv
from agents import load_base_model, create_agents
from trainer import GRPOTrainer, TrainerConfig
from utils import setup_logging, TrainingMetrics, LanguageAnalyzer

setup_logging(level='INFO')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


PyTorch: 2.10.0+cu128
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB


In [ ]:
# ── Configuración de hardware ─────────────────────────────────────────────────

# Detecta automáticamente el mejor dispositivo disponible
if torch.cuda.is_available():
    DEVICE = 'cuda'
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_4BIT = VRAM_GB < 6  # 4-bit si menos de 6GB VRAM
    print(f'Usando GPU ({VRAM_GB:.1f} GB VRAM). 4-bit: {USE_4BIT}')
else:
    DEVICE = 'cpu'
    USE_4BIT = False
    print('Usando CPU (más lento, pero funciona)')

# ── Selección del modelo ──────────────────────────────────────────────────────
# Opciones (de más ligero a más capaz):
#   'Qwen/Qwen2.5-0.5B-Instruct'  — ~1GB RAM/VRAM (muy rápido)
#   'Qwen/Qwen2.5-1.5B-Instruct'  — ~3GB RAM/VRAM (recomendado)
#   'Qwen/Qwen2.5-3B-Instruct'    — ~6GB RAM (solo si tienes suficiente)
#   'microsoft/Phi-3-mini-4k-instruct' — ~8GB RAM (alternativa)

# MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'

# Para 4GB VRAM usa 0.5B:
# MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

# Para llama3.2 3B (si tienes 6GB+ VRAM):
# MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'

# MODEL_NAME = 'unsloth/Llama-3.2-1B-bnb-4bit'

MODEL_NAME = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"

# MODEL_NAME = "unsloth/phi-2"

Usando GPU (4.3 GB VRAM). 4-bit: True


In [9]:
# ── [OPCIONAL] Acelerar con Unsloth ──────────────────────────────────────────
# Descomenta si tienes Unsloth instalado (solo GPU)

USE_UNSLOTH = True

if USE_UNSLOTH and DEVICE == 'cuda':
    from unsloth import FastLanguageModel, get_chat_template
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=USE_4BIT,
        trust_remote_code=True,
    )
    base_model = FastLanguageModel.get_peft_model(
        base_model,
        r=8,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        lora_alpha=32,
        lora_dropout=0.1,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", 
    # chat_template = "llama-3", 
    )
    print('Modelo cargado con Unsloth ✓')
else:
    print('Cargando modelo estándar (sin Unsloth)...')

2026-04-15 17:57:39,146 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unsloth/Phi-3-mini-4k-instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 17:57:39,240 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Phi-3-mini-4k-instruct-bnb-4bit/81453e5718775630581ab9950e6c0ccf0d7a4177/config.json "HTTP/1.1 200 OK"
2026-04-15 17:57:39,297 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Phi-3-mini-4k-instruct-bnb-4bit/81453e5718775630581ab9950e6c0ccf0d7a4177/config.json "HTTP/1.1 200 OK"
2026-04-15 17:57:39,433 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unsloth/Phi-3-mini-4k-instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


2026-04-15 17:57:40,871 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/unslothai/other/revision/main "HTTP/1.1 200 OK"
2026-04-15 17:57:41,031 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unslothai/other/resolve/43d9e0f2f19a5d7836895f648dc0e762816acf77/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 17:57:41,055 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/other/43d9e0f2f19a5d7836895f648dc0e762816acf77/config.json "HTTP/1.1 200 OK"
2026-04-15 17:57:41,082 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/other/43d9e0f2f19a5d7836895f648dc0e762816acf77/config.json "HTTP/1.1 200 OK"
2026-04-15 17:57:41,111 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unslothai/other/resolve/43d9e0f2f19a5d7836895f648dc0e762816acf77/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-15 17:57:41,117 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unslothai/other/res

Modelo cargado con Unsloth ✓


In [10]:
# ── Carga del modelo base ─────────────────────────────────────────────────────

if not USE_UNSLOTH:
    base_model, tokenizer = load_base_model(
        model_name=MODEL_NAME,
        device='auto' if DEVICE == 'cuda' else 'cpu',
        use_4bit=USE_4BIT,
        use_flash_attention=False,  # ponlo True si tienes GPU Ampere+
    )
    print(f'Modelo {MODEL_NAME} cargado ✓')

print(f'\nParámetros totales: {base_model.num_parameters():,}')


Parámetros totales: 3,827,371,008


In [12]:
# ── Crear agentes ─────────────────────────────────────────────────────────────
# shared_lora=True:  todos aprenden juntos (un solo adaptador)
# shared_lora=False: cada agente tiene su propio adaptador LoRA
#                    (más memoria, pero permite especialización)

NUM_PLAYERS = 4
SHARED_LORA = True  # recomendado para 4GB VRAM
LORA_R = 12          # rango LoRA. Más alto = más parámetros (max 16 para hardware limitado)

agents = create_agents(
    model=base_model,
    tokenizer=tokenizer,
    num_players=NUM_PLAYERS,
    device=DEVICE,
    lora_r=LORA_R,
    shared_lora=SHARED_LORA,
)

print(f'{len(agents)} agentes creados ✓')
print(f'LoRA compartida: {SHARED_LORA}')
agents[0].model.print_trainable_parameters()

trainable params: 9,437,184 || all params: 3,830,516,736 || trainable%: 0.2464
4 agentes creados ✓
LoRA compartida: True
trainable params: 9,437,184 || all params: 3,830,516,736 || trainable%: 0.2464


In [13]:
# ── SFT: entrenamiento supervisado previo al RL ───────────────────────────────
from sft_trainer import run_sft, verify_sft_quality

print("Iniciando SFT...")
sft_results = run_sft(
    agents=agents,
    tokenizer=tokenizer,
    dataset_path="sft_dataset.json",
    epochs=3,           # 3-5 épocas suele ser suficiente con 250 ejemplos
    batch_size=2,       # reducir a 1 si hay OOM en GPU 4GB
    lr=2e-4,
    max_length=512,
    save_dir="checkpoints/sft",
    device=DEVICE,
    shared_lora=SHARED_LORA,
)

2026-04-15 17:59:01,214 [INFO] sft_trainer: Cargados 268 ejemplos de SFT desde sft_dataset.json
2026-04-15 17:59:01,353 [INFO] sft_trainer: Dataset SFT: 268 ejemplos listos
2026-04-15 17:59:01,370 [INFO] sft_trainer: Iniciando SFT — modelo compartido
`use_return_dict` is deprecated! Use `return_dict` instead!


Iniciando SFT...
Unsloth: Will smartly offload gradients to save VRAM!


2026-04-15 18:05:39,329 [INFO] sft_trainer:   Epoch 1/3 — loss: 0.7705 | lr: 1.55e-04


  [compartido] Epoch 1/3 — loss: 0.7705


2026-04-15 18:10:47,199 [INFO] sft_trainer:   Epoch 2/3 — loss: 0.4090 | lr: 6.50e-05


  [compartido] Epoch 2/3 — loss: 0.4090


2026-04-15 18:15:55,500 [INFO] sft_trainer:   Epoch 3/3 — loss: 0.2784 | lr: 2.00e-05


  [compartido] Epoch 3/3 — loss: 0.2784


2026-04-15 18:15:56,347 [INFO] sft_trainer: Adaptador SFT guardado en checkpoints/sft/shared



SFT completado. Loss final: 0.2784


In [14]:
# ── Verificar calidad del SFT antes de pasar al RL ────────────────────────────
json_ok_rate = verify_sft_quality(agents, tokenizer, num_samples=5, device=DEVICE)

if json_ok_rate >= 0.8:
    print(f"\n✓ SFT exitoso ({json_ok_rate:.0%} JSON válidos). Listo para RL.")
else:
    print(f"\n⚠ Solo {json_ok_rate:.0%} JSON válidos.")
    print("  Opciones: aumentar epochs, reducir lr, o continuar igualmente.")


VERIFICACIÓN POST-SFT


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 1 ✓
  Mano: [4] | Mesa: 0 | Vidas: 3
  Generado: {"msg": "espero bastante", "act": "wait"}
  Parseado → act=wait, msg='espero bastante'


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 2 ✓
  Mano: [45] | Mesa: 0 | Vidas: 3
  Generado: {"msg": "espero bastante", "act": "wait"}
  Parseado → act=wait, msg='espero bastante'


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 3 ✓
  Mano: [12] | Mesa: 10 | Vidas: 2
  Generado: {"msg": "yo también pronto", "act": "play"}
  Parseado → act=play, msg='yo también pronto'


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 4 ✓
  Mano: [33] | Mesa: 10 | Vidas: 2
  Generado: {"msg": "yo mucho después", "act": "wait"}
  Parseado → act=wait, msg='yo mucho después'

Ejemplo 5 ✓
  Mano: [7] | Mesa: 5 | Vidas: 1
  Generado: {"msg": "voy ya", "act": "play"}
  Parseado → act=play, msg='voy ya'

Resultado: 5/5 respuestas con JSON válido

✓ SFT exitoso (100% JSON válidos). Listo para RL.


In [15]:
# ── Test rápido: una partida sin entrenamiento ─────────────────────────────────
print('Ejecutando partida de prueba...')

env = TheMindEnv(num_players=NUM_PLAYERS)
state = env.reset(level=1)

print(f'Manos iniciales:')
for pid, hand in state.hands.items():
    print(f'  Jugador {pid}: {hand}')

# Probar un turno
obs = env.get_observation(0)
decision = agents[0].generate_action(obs)

print(f'\nDecisión del agente 0:')
print(f'  Mensaje:   {decision["message"]}')
print(f'  Acción:    {decision["action"]}')
print(f'  Razonamiento: {decision["reasoning"][:100]}...')

2026-04-15 18:16:38,244 [INFO] environment: Nueva ronda. Nivel 1. Manos: {0: [26], 1: [24], 2: [20], 3: [12]}
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'top_p', 'do_sample', 'cache_implementation', 'max_new_tokens', 'pad_token_id', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ejecutando partida de prueba...
Manos iniciales:
  Jugador 0: [26]
  Jugador 1: [24]
  Jugador 2: [20]
  Jugador 3: [12]

Decisión del agente 0:
  Mensaje:   {
  Acción:    play
  Razonamiento: {
   "message": "", // No voy porque soy yo uno solo justo aquí; espero tranquilo por mi parte
   "a...


In [16]:
# ── Configuración del entrenamiento ───────────────────────────────────────────

config = TrainerConfig(
    num_episodes=250,          # empieza con 200 para ver si funciona
    num_levels=8,              # niveles 1 y 2
    group_size=4,              # 4 juegos por actualización GRPO
    lr=1e-5,                   # LR conservador para LLMs pequeños
    kl_coeff=0.01,
    clip_ratio=0.2,
    entropy_bonus=0.01,
    warmup_episodes=10,        # los primeros 20 ep solo exploración
    max_turns_per_episode_ratio=25,
    messages_per_turn=True,
    device=DEVICE,
    accumulate_grad_steps=4,   # simula batch de 4 sin más memoria
    checkpoint_every=5,
)

# Para un entrenamiento más largo y serio:
# config.num_episodes = 1000
# config.num_levels = 3

print('Configuración lista ✓')
print(f'  Episodios: {config.num_episodes}')
print(f'  LR: {config.lr}')
print(f'  Group size (GRPO): {config.group_size}')

Configuración lista ✓
  Episodios: 250
  LR: 1e-05
  Group size (GRPO): 4


In [17]:
# ── Inicializar entrenador ────────────────────────────────────────────────────

metrics = TrainingMetrics()
lang_analyzer = LanguageAnalyzer()

# Hace la celda autosuficiente si no ejecutaste la celda de test.
if 'env' not in globals() or env is None:
    env = TheMindEnv(num_players=NUM_PLAYERS)

trainer = GRPOTrainer(
    agents=agents,
    env=env,
    config=config,
)

print('Entrenador GRPO listo ✓')

Entrenador GRPO listo ✓


In [ ]:
# ── ENTRENAMIENTO ─────────────────────────────────────────────────────────────
# ¡Esta celda puede tardar bastante!
#   - CPU 12GB Qwen2.5-1.5B: ~30-60 seg/episodio
#   - GPU 4GB Qwen2.5-1.5B:  ~5-15 seg/episodio

print('Iniciando entrenamiento...')
print('(Ctrl+C para detener y ver resultados parciales)')

if USE_UNSLOTH:
    os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

try:
    trainer.train(
        metrics=metrics,
        lang_analyzer=lang_analyzer,
        verbose=True,
        very_verbose=True
    )
except KeyboardInterrupt:
    print('\nEntrenamiento interrumpido por el usuario.')

2026-04-15 18:21:07,772 [INFO] environment: Nueva ronda. Nivel 1. Manos: {0: [27], 1: [31], 2: [24], 3: [19]}
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iniciando entrenamiento...
(Ctrl+C para detener y ver resultados parciales)


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turno 1 | Player 0 | Obs: {'player_id': 0, 'my_hand': [27], 'played_cards': [], 'table_top': 0, 'lives': 1, 'stars': 0, 'messages': [], 'level': 1, 'num_players': 4} | Message: {


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Turno 1 | Player 3 | Obs: {'player_id': 3, 'my_hand': [42], 'played_cards': [], 'table_top': 0, 'lives': 1, 'stars': 0, 'messages': [{'player': 2, 'text': '{'}], 'level': 1, 'num_players': 4} | Message: 


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-04-15 18:27:01,619 [INFO] environment: Error: jugador 1 jugó 16 sobre 17. Vidas: 0
2026-04-15 18:27:01,620 [INFO] environment: Nueva ronda. Nivel 1. Manos: {0: [16], 1: [49], 2: [29], 3: [39]}
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turno 1 | Player 0 | Obs: {'player_id': 0, 'my_hand': [16], 'played_cards': [], 'table_top': 0, 'lives': 1, 'stars': 0, 'messages': [], 'level': 1, 'num_players': 4} | Message: 


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turno 1 | Player 1 | Obs: {'player_id': 1, 'my_hand': [49], 'played_cards': [], 'table_top': 0, 'lives': 1, 'stars': 0, 'messages': [], 'level': 1, 'num_players': 4} | Message: 


In [ ]:
# ── Análisis de resultados ────────────────────────────────────────────────────

metrics.print_summary()
metrics.plot(f'training_metrics_final_{MODEL_NAME}.png')

lang_analyzer.print_report()
lang_analyzer.save_log(f'language_log_{MODEL_NAME}.json')


RESUMEN DE ENTRENAMIENTO
Episodios totales: 44
Win rate (último 50): 0.00%
Reward medio (último 50): -7.670
Errores medio (último 50): 0.18

Lenguaje más reciente (ep 8):
  Palabras más comunes: [('message', 3), ('action', 3), ('wait', 2), ('reasoning', 2), ('esperando', 1)]
  Longitud media msg:   69.7 chars
Métricas guardadas en training_metrics_final_qwen2.5-1.5B.png

ANÁLISIS DEL LENGUAJE EMERGENTE
Total de mensajes analizados: 54

Estrategias detectadas:
  señal: 25 ocurrencias
  urgencia: 18 ocurrencias
  calma: 18 ocurrencias
  acuerdo: 1 ocurrencias
  duda: 1 ocurrencias

Evolución del vocabulario:
  Fase inicio: ['message', 'action', 'reasoning', 'wait', 'espero']
  Fase medio: ['message', 'action', 'reasoning', 'wait', 'espero']
  Fase final: ['message', 'action', 'reasoning', 'wait', 'espero']
  Fase final: ['message', 'action', 'wait', 'reasoning', 'esperando']
Log guardado en language_log_qwen2.5-1.5B.json


In [ ]:
# ── Ver ejemplos de mensajes recientes ───────────────────────────────────────
import json

print('Últimos 20 mensajes de los agentes:\n')
for msg in lang_analyzer.message_log[-20:]:
    print(f'  Ep {msg["episode"]:3d} | J{msg["player"]}: {msg["message"]}')

Últimos 20 mensajes de los agentes:

  Ep  19 | J0: la descalificación
  Ep  19 | J1: afectarte.
  Ep  19 | J2: lo a ti
  Ep  19 | J0: .
  Ep  19 | J1: a.
  Ep  19 | J1: urgente
  Ep  19 | J2: {"urgency": "yo soy el último", "puede ser yo también", "espero tranquilo"}
  Ep  19 | J0: {"urgency":"zona media","y":{"puede ser yo también"},{seré yo primero}}
  Ep  19 | J1: {"urgency": "urgente", "yo soy el último", "zona media", "voy justo ahora", "yo espero tranquilament
  Ep  19 | J2: {"urgency": "urgente", "yo soy el último", "zona media", "voy justo ahora", "yo espero tranquilo"}
  Ep  19 | J3: decisión.
  Ep  19 | J3: .
  Ep  19 | J0: as algo sobre ti mismo.
  Ep  19 | J1: as como la oportunidad perfecta para actuar.
  Ep  19 | J2: arte así:
  Ep  19 | J3: mucho, solo necesitas saber dónde están tus cartas.
  Ep  19 | J2: eligencia humana)"
  Ep  19 | J3: es darle un nombre a cada uno de estos tres estados:
  Ep  19 | J0: expectativa o confianza.
  Ep  19 | J2: .


In [ ]:
# ── Vocabulario emergente por fases ───────────────────────────────────────────

vocab = lang_analyzer.get_vocabulary_evolution(n_bins=3)

fase_names = ['Inicio', 'Medio', 'Final']
for i, (bin_idx, words) in enumerate(vocab.items()):
    print(f'\nFase {fase_names[min(i, 2)]}')
    print('Top 10 palabras:')
    for word, count in words[:10]:
        bar = '█' * min(count, 30)
        print(f'  {word:15s} {bar} ({count})')


Fase Inicio
Top 10 palabras:
  yo              ██████████████████████████████ (165)
  urgency         ██████████████████████████████ (142)
  soy             ██████████████████████████████ (57)
  urgente         ██████████████████████████████ (47)
  y               ██████████████████████████████ (43)
  algo            ██████████████████████████████ (37)
  ser             ██████████████████████████████ (35)
  espero          ██████████████████████████████ (33)
  el              ██████████████████████████████ (33)
  tengo           ██████████████████████████████ (33)

Fase Medio
Top 10 palabras:
  yo              ██████████████████████████████ (166)
  urgency         ██████████████████████████████ (158)
  soy             ██████████████████████████████ (59)
  urgente         ██████████████████████████████ (47)
  el              ██████████████████████████████ (45)
  zona            ██████████████████████████████ (41)
  ser             ██████████████████████████████ (38)
  voy             █

In [ ]:
# ── Jugar una partida con los agentes entrenados ──────────────────────────────

print('Partida de demostración con agentes entrenados')
print('='*60)

state = env.reset(level=2)  # nivel 2: 2 cartas por jugador

print('Manos (secretas en el juego real):')
for pid, hand in state.hands.items():
    print(f'  J{pid}: {hand}')
print()

turn = 0
max_turns = 50

while not env.is_done() and turn < max_turns:
    turn += 1
    print(f'--- Turno {turn} | Carta en la mesa: {state.table_top} | Vidas: {state.lives} ---')

    for agent in agents:
        if not state.hands[agent.player_id]:
            continue

        obs = env.get_observation(agent.player_id)
        decision = agent.generate_action(obs)

        if decision['message']:
            env.send_message(agent.player_id, decision['message'])
            print(f'  J{agent.player_id} dice: "{decision["message"]}')

        if decision['action'] == 'play':
            card = agent.get_card_to_play(obs)
            if card:
                result = env.play_card(agent.player_id, card)
                icon = '✓' if result.get('correct') else '✗'
                print(f'  J{agent.player_id} juega {card} {icon}')

        if env.is_done():
            break

print()
if state.won or state.round_over:
    print('🏆 ¡Victoria! Cartas jugadas en orden:', state.played_cards)
else:
    print('💀 Derrota. Cartas jugadas:', state.played_cards)

2026-04-11 14:45:36,828 [INFO] environment: Nueva ronda. Nivel 2. Manos: {0: [8, 23], 1: [5, 39], 2: [11, 47], 3: [28, 31]}
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Partida de demostración con agentes entrenados
Manos (secretas en el juego real):
  J0: [8, 23]
  J1: [5, 39]
  J2: [11, 47]
  J3: [28, 31]

--- Turno 1 | Carta en la mesa: 0 | Vidas: 1 ---


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  J0 dice: "urgente
  J0 juega 8 ✓


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

--- Turno 49 | Carta en la mesa: 8 | Vidas: 1 ---


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

--- Turno 50 | Carta en la mesa: 8 | Vidas: 1 ---


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



💀 Derrota. Cartas jugadas: [8]


In [ ]:
# ── Guardar modelo final ──────────────────────────────────────────────────────

from utils import save_checkpoint

save_checkpoint(
    agents=agents,
    metrics=metrics,
    episode=config.num_episodes,
    output_dir='checkpoints/final_qwen2.5-1.5B',
)
print('Modelo guardado ✓')

Checkpoint guardado en: checkpoints2/episode_1500
  Agente 0: 8.76 MB
  Agente 1: 8.76 MB
  Agente 2: 8.76 MB
  Agente 3: 8.76 MB
Modelo guardado ✓


In [ ]:
# ── Reanudar entrenamiento desde un checkpoint ────────────────────────────────

from utils import list_checkpoints, load_checkpoint

# 1. Ver qué checkpoints tienes disponibles
list_checkpoints("checkpoints")

# 2. Cargar el modelo base y crear los agentes igual que siempre (Ejecutar arriba mejor)
# base_model, tokenizer = load_base_model(model_name=MODEL_NAME, ...)
# agents = create_agents(base_model, tokenizer, ...)

  Episodio   Win rate     Reward    Errores
--------------------------------------------
       250      0.00%       0.00       0.00
      1500      0.00%     -10.66       1.00


[{'episode': 250, 'path': 'checkpoints2/episode_250'},
 {'episode': 1500,
  'path': 'checkpoints2/episode_1500',
  'win_rate': 0.0,
  'avg_reward': -10.658000000000001,
  'mistakes': 1.0}]

In [ ]:
# Crear también los optimizadores antes de cargar (para restaurar su estado)
from torch.optim import AdamW
optimizers = [AdamW(a.model.parameters(), lr=1e-5) for a in agents]

# 3. Cargar el checkpoint — devuelve las métricas restauradas
RESUME_FROM_EPISODE = 250
metrics = load_checkpoint(
    agents=agents,
    episode=RESUME_FROM_EPISODE,
    output_dir="checkpoints2",
    optimizers=optimizers,   # opcional, pero recomendado
)

# 4. Crear el trainer pasando los optimizadores ya cargados
trainer = GRPOTrainer(agents=agents, env=env, config=config, optimizers=optimizers)
trainer.episode_count = RESUME_FROM_EPISODE  # que sepa desde dónde va

# 5. Ajustar config para entrenar los episodios restantes
config.num_episodes = 1500  # episodios TOTALES, no adicionales
# El scheduler de niveles se recalculará, así que tenlo en cuenta

# 6. Reanudar
print('Reanudando entrenamiento...')
print('(Ctrl+C para detener y ver resultados parciales)')

try:
    trainer.train(
        metrics=metrics,
        lang_analyzer=lang_analyzer,
        verbose=True,
    )
except KeyboardInterrupt:
    print('\nEntrenamiento interrumpido por el usuario.')

  Agente 0 cargado desde checkpoints2/episode_250/agent_0
  Agente 1 cargado desde checkpoints2/episode_250/agent_1


2026-04-10 18:47:23,374 [INFO] environment: Nueva ronda. Nivel 1. Manos: {0: [26], 1: [24], 2: [20], 3: [12]}


  Agente 2 cargado desde checkpoints2/episode_250/agent_2
  Agente 3 cargado desde checkpoints2/episode_250/agent_3
  AVISO: no se encontró estado del optimizador 0
  AVISO: no se encontró estado del optimizador 1
  AVISO: no se encontró estado del optimizador 2
  AVISO: no se encontró estado del optimizador 3

Checkpoint ep 250 cargado. Listo para reanudar.
Reanudando entrenamiento...
(Ctrl+C para detener y ver resultados parciales)


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Ep    0 | Nivel 1 | Win: 0.00 | Reward: -11.03 | Errores: 1.0


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Ep   10 | Nivel 1 | Win: 0.00 | Reward: -10.63 | Errores: 1.0


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

NotImplementedError: Unsloth: Logits are empty from 2024.11 onwards. To get raw logits again, please set the environment variable `UNSLOTH_RETURN_LOGITS` to `"1" BEFORE starting to train ie before `trainer.train()`. For example:
```
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
trainer.train()
```
No need to restart your console - just add `os.environ['UNSLOTH_RETURN_LOGITS'] = '1'` before trainer.train() and re-run the cell!